In [ ]:
# Proximity Analysis on different types of attributes
#
# In this lab, we'll primarily cover performing proximity analysis on 4
# different kinds of attributes:
#
# 1. Nominal Attributes
# 2. Binary Attributes
#    a. Simple Matching Co-efficient (SMC) for symmetric binary attribute
#    b. Jaccard Coefficient for asymmetric binary attribute
# 3. Numeric Attributes
#    a. Manhattan distance
#    b. Euclidean distance
#    c. Supremum distance
# 4. Ordinal Attributes (standard and weighted)

In [ ]:

# Download the dataset file to local storage to current directory (from publicly
# shared drive file)
#
# Original source:
# https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques
!gdown 1Yhw8tBCmjYrDZ5Rm1xPrP2y5bmBmxoEP

Downloading...
From: https://drive.google.com/uc?id=1Yhw8tBCmjYrDZ5Rm1xPrP2y5bmBmxoEP
To: /content/housing.csv
100% 461k/461k [00:00<00:00, 28.0MB/s]


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import pairwise_distances

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
# Load the dataset file into memory
DATASET_PATH = "/content/housing.csv"
df = pd.read_csv(DATASET_PATH)

# For this experiment, to keep the output more simpler of analysis we'll stick
# using only the first N rows
N = 10
df_head = df.head(N)
df_head

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
5,6,50,RL,85.0,14115,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,MnPrv,Shed,700,10,2009,WD,Normal,143000
6,7,20,RL,75.0,10084,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,307000
7,8,60,RL,NaN,10382,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,Shed,350,11,2009,WD,Normal,200000
8,9,50,RM,51.0,6120,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2008,WD,Abnorml,129900
9,10,190,RL,50.0,7420,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,1,2008,WD,Normal,118000


In [ ]:
# 1. Nominal attributes
#
# Nominal attributes are attributes that can hold multiple values with distinct
# meanings

In [ ]:
# Explore different attributes present in the dataset to choose one for analysis
df_head.select_dtypes(include=['object']).columns

Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='object')

In [ ]:
nominal_column_name = "LotShape"
nominal_attribute_values = df_head[nominal_column_name]
nominal_attribute_values

,LotShape
0,Reg
1,Reg
2,IR1
3,IR1
4,IR1
5,IR1
6,Reg
7,IR1
8,Reg
9,Reg


In [ ]:
# Select a nominal attribute for similarity comparison for single attribute
nominal_column_name = "LotShape"
nominal_attribute_values = df_head[nominal_column_name]
ids = df_head.index

# Get the row count to initialize
n = len(nominal_attribute_values)

# Initialize an empty numpy array for the similarity matrix
# We use a float type to hold the similarity scores (0.0 or 1.0)
similarity_matrix = np.zeros((n, n), dtype=float)

# Use a nested loop to compare each item to every other item
for i in range(n):
    for j in range(n):
        # Sets 1.0 if true else 0.0 (bool -> equivalent int conversion)
        similarity_matrix[i, j] = int(nominal_attribute_values[i]
                                      == nominal_attribute_values[j])

similarity_df = pd.DataFrame(similarity_matrix, index=ids, columns=ids)
similarity_df

# For multiple attributes, simple aggregation can be done based on the
# requirements of the analysis

,0,1,2,3,4,5,6,7,8,9
0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0
1,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0
2,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
3,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
4,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
5,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
6,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0
7,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
8,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0
9,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0


In [ ]:
# 2. Binary Attributes
#
# Binary attributes are attributes that can two values often complementary in
# nature
#
# They can be classified into:
#
# 1. Symmetric (both values are significant)
# 2. Asymmetric (only one kind of value is significant)
#
# In case of symmetric attributes, we consider both positive/negative matches
# for similarity. However in case of asymmetric, we only consider the
# positive/significant class as two entities having negative/insignificant
# class may not imply they are the same/similar
#
# Significance or the classification is very specific to the context of analysis
#
# We'll be dealing with both to understand how they can be processed for
# analysis

In [ ]:
# Explore different attributes present in the dataset to choose one for analysis
df_head.select_dtypes(include=['bool']).columns

Index([], dtype='object')

In [ ]:
df["CentralAir"].unique()

array(['Y', 'N'], dtype=object)

In [ ]:
# We have directly loaded the data without any pre-processing, therefore we will
# need to pre-process required columns from the dataset for this purpose
#
# For the purpose of demonstration, we'll be choosing a symmetric and asymmetric
# binary attribute
#
# CentralAir can be considered as a symmetric attribute
symmetric_column_name = "CentralAir"
symmetric_column_values = df_head[symmetric_column_name]

# GarageType can be considered as an asymmetric attribute
asymmetric_column_values = np.where(df_head['SaleCondition'] == 'Normal', True, False)

# ...
ids = df_head.index

set(asymmetric_column_values)

{np.False_, np.True_}

In [ ]:
# a. Simple Matching Co-efficient (SMC) for symmetric binary attribute

# Get the row count to initialize
n = len(symmetric_column_values)

# Initialize an empty numpy array for the similarity matrix
# We use a float type to hold the similarity scores (0.0 or 1.0)
similarity_matrix = np.zeros((n, n), dtype=float)

# Use a nested loop to compare each item to every other item
for i in range(n):
    for j in range(n):
        # Sets 1.0 if true else 0.0 (bool -> equivalent int conversion)
        similarity_matrix[i, j] = int(symmetric_column_values[i]
                                      == symmetric_column_values[j])

similarity_df = pd.DataFrame(similarity_matrix, index=ids, columns=ids)
similarity_df

,0,1,2,3,4,5,6,7,8,9
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
5,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
6,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
9,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [ ]:
# b. Jaccard Coefficient for asymmetric binary attribute

# Get the row count to initialize
n = len(asymmetric_column_values)

# Initialize an empty numpy array for the similarity matrix
# We use a float type to hold the similarity scores (0.0 or 1.0)
similarity_matrix = np.zeros((n, n), dtype=float)

# Use a nested loop to compare each item to every other item
for i in range(n):
    for j in range(n):
        # Checks if its the same and if the value is true and sets the similarity matrix accordingly
        if asymmetric_column_values[i] == asymmetric_column_values[j] and asymmetric_column_values[j] == True:
          similarity_matrix[i, j] = 1

similarity_df = pd.DataFrame(similarity_matrix, index=ids, columns=ids)
similarity_df

,0,1,2,3,4,5,6,7,8,9
0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0
1,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0
2,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0
5,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0
6,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0
7,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0


In [ ]:
# 3. Numeric Attributes
#
# Numeric attributes are attributes that hold any real number
#

# We have mainly performed various kinds of proximity analysis using Minkowski
# distance

In [ ]:
# Normalize the numerical data
scaler = MinMaxScaler()
numeric_df = df_head.select_dtypes(include=['number']).dropna()
df_normalized = scaler.fit_transform(numeric_df)
numeric_df

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
0,1,60,65.0,8450,7,5,2003,2003,196.0,706,...,0,61,0,0,0,0,0,2,2008,208500
1,2,20,80.0,9600,6,8,1976,1976,0.0,978,...,298,0,0,0,0,0,0,5,2007,181500
2,3,60,68.0,11250,7,5,2001,2002,162.0,486,...,0,42,0,0,0,0,0,9,2008,223500
3,4,70,60.0,9550,7,5,1915,1970,0.0,216,...,0,35,272,0,0,0,0,2,2006,140000
4,5,60,84.0,14260,8,5,2000,2000,350.0,655,...,192,84,0,0,0,0,0,12,2008,250000
5,6,50,85.0,14115,5,5,1993,1995,0.0,732,...,40,30,0,320,0,0,700,10,2009,143000
6,7,20,75.0,10084,8,5,2004,2005,186.0,1369,...,255,57,0,0,0,0,0,8,2007,307000
8,9,50,51.0,6120,7,5,1931,1950,0.0,0,...,90,0,205,0,0,0,0,4,2008,129900
9,10,190,50.0,7420,5,6,1939,1950,0.0,851,...,0,4,0,0,0,0,0,1,2008,118000


In [ ]:
# a. Manhattan distance
#
# Using Minkowski with p=1 is as good as calculating Manhattan distance
distance_matrix = pairwise_distances(df_normalized, metric='minkowski', p=1)

# Convert the NumPy array back to a DataFrame for readability
distance_df = pd.DataFrame(distance_matrix, index=numeric_df.index, columns=numeric_df.index)
distance_df

,0,1,2,3,4,5,6,8,9
0,0.000000,13.037220,3.553917,9.745880,8.367292,10.769207,9.327657,13.653465,15.698626
1,13.037220,0.000000,12.004733,13.651844,15.382889,15.491474,10.115071,13.871509,14.394546
2,3.553917,12.004733,0.000000,8.996070,6.706937,9.963795,8.115733,13.261133,15.745157
3,9.745880,13.651844,8.996070,0.000000,13.532983,13.678200,12.742517,10.494026,13.215387
4,8.367292,15.382889,6.706937,13.532983,0.000000,13.788304,9.427632,17.374888,21.135804
5,10.769207,15.491474,9.963795,13.678200,13.788304,0.000000,15.269918,18.145073,15.071984
6,9.327657,10.115071,8.115733,12.742517,9.427632,15.269918,0.000000,16.264077,17.395630
8,13.653465,13.871509,13.261133,10.494026,17.374888,18.145073,16.264077,0.000000,10.403327
9,15.698626,14.394546,15.745157,13.215387,21.135804,15.071984,17.395630,10.403327,0.000000


In [ ]:
# b. Euclidean distance
#
# Using Minkowski with p=1 is as good as calculating for Euclidean distance
distance_matrix = pairwise_distances(df_normalized, metric='minkowski', p=2)

# Convert the NumPy array back to a DataFrame for readability
distance_df = pd.DataFrame(distance_matrix, index=numeric_df.index, columns=numeric_df.index)
distance_df

,0,1,2,3,4,5,6,8,9
0,0.000000,2.948437,1.140258,2.533854,2.032785,2.656317,2.465196,3.250209,3.443055
1,2.948437,0.000000,2.811036,3.070923,3.348885,3.334131,2.528081,3.051787,3.192210
2,1.140258,2.811036,0.000000,2.463289,1.707875,2.402836,2.261849,3.039886,3.311501
3,2.533854,3.070923,2.463289,0.000000,3.090242,3.117807,3.063605,2.533847,2.922281
4,2.032785,3.348885,1.707875,3.090242,0.000000,3.208151,2.255006,3.643237,4.285106
5,2.656317,3.334131,2.402836,3.117807,3.208151,0.000000,3.342725,3.833375,3.349819
6,2.465196,2.528081,2.261849,3.063605,2.255006,3.342725,0.000000,3.466191,3.597031
8,3.250209,3.051787,3.039886,2.533847,3.643237,3.833375,3.466191,0.000000,2.672689
9,3.443055,3.192210,3.311501,2.922281,4.285106,3.349819,3.597031,2.672689,0.000000


In [ ]:
# c. Supremum distance
#
# Using Minkowski with p=infinity is as good as calculating   distance
distance_matrix = pairwise_distances(df_normalized, metric='minkowski', p=np.inf)

# Convert the NumPy array back to a DataFrame for readability
distance_df = pd.DataFrame(distance_matrix, index=numeric_df.index, columns=numeric_df.index)
distance_df

In [ ]:
# Ordinal attributes
#
# Ordinal attributes are those attributes that have an meaningful order or rank
#
# Such attributes could naturally be present in the dataset which in our case
# is the "OverallQual" attribute/column. Its range is whole number upto 10.
#
# If its not present, it can be derived from one or more other attributes
#
# In cases where each rank needs to be given a different weightage, the ordinal
# attribute before performing proximity analysis could be scaled and varied
# as per requirements
#
# For e.g. Let's say we have an ordinary sequence 1,2,3,4,....
# We want to give more weightage to the starting three ranks, so we can increase
# the difference between 1-2 and 2-3 such that the new values are:
# 1,5,10,11,12,13,... (using their index/row number as reference)

In [ ]:
# We'll consider the "OverallQual" column for proximity analysis
overall_qual_column = df_head["OverallQual"]
overall_qual_column

,OverallQual
0,7
1,6
2,7
3,7
4,8
5,5
6,8
7,7
8,7
9,5


In [ ]:
# Normalize the data
scaler = MinMaxScaler()
overall_qual = df_head[["OverallQual"]].values
overall_qual_normalized = scaler.fit_transform(overall_qual)

In [ ]:
# Euclidean distance
distance_matrix = pairwise_distances(overall_qual_normalized, metric='minkowski', p=2)

# Convert the NumPy array back to a DataFrame for readability
distance_df = pd.DataFrame(distance_matrix, index=df_head.index, columns=df_head.index)
distance_df

,0,1,2,3,4,5,6,7,8,9
0,0.000000,0.333333,0.000000,0.000000,0.333333,0.666667,0.333333,0.000000,0.000000,0.666667
1,0.333333,0.000000,0.333333,0.333333,0.666667,0.333333,0.666667,0.333333,0.333333,0.333333
2,0.000000,0.333333,0.000000,0.000000,0.333333,0.666667,0.333333,0.000000,0.000000,0.666667
3,0.000000,0.333333,0.000000,0.000000,0.333333,0.666667,0.333333,0.000000,0.000000,0.666667
4,0.333333,0.666667,0.333333,0.333333,0.000000,1.000000,0.000000,0.333333,0.333333,1.000000
5,0.666667,0.333333,0.666667,0.666667,1.000000,0.000000,1.000000,0.666667,0.666667,0.000000
6,0.333333,0.666667,0.333333,0.333333,0.000000,1.000000,0.000000,0.333333,0.333333,1.000000
7,0.000000,0.333333,0.000000,0.000000,0.333333,0.666667,0.333333,0.000000,0.000000,0.666667
8,0.000000,0.333333,0.000000,0.000000,0.333333,0.666667,0.333333,0.000000,0.000000,0.666667
9,0.666667,0.333333,0.666667,0.666667,1.000000,0.000000,1.000000,0.666667,0.666667,0.000000
